# Notebook 08 — Streaming Runtime Adaptation

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 07 created an adaptive execution selector from static distribution-level metrics.

Notebook 08 extends that selector into a streaming/runtime setting:

- split a data stream into windows,
- estimate local structure per window,
- select an execution policy per window,
- simulate switching costs,
- compare adaptive policy vs fixed scalar/SIMD baselines.

Constraint view:
> runtime adaptation asks whether coherence persists as distributions drift.

## Goals

1. Generate or load integer streams with changing regimes.
2. Window each stream.
3. Compute local structure metrics per window.
4. Select execution policy per window:
   - scalar
   - SIMD
   - coherent-local
   - guarded fallback
   - hybrid
5. Simulate throughput with switching cost.
6. Compare:
   - fixed scalar
   - fixed SIMD
   - adaptive selector
7. Export CSV, JSON, Markdown report, and PNG figures.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Generate synthetic regime-switching streams

Each stream combines several regimes:

- low-entropy repeating
- sequential IDs
- uniform 32-bit
- zipfian smallints
- clustered ranges

Later versions can replace this with real traces from serialization workloads.

In [ ]:
rng = np.random.default_rng(42)

def gen_low_entropy(n):
    pattern = np.array([1, 2, 3, 4], dtype=np.int64)
    return np.tile(pattern, int(np.ceil(n / len(pattern))))[:n]

def gen_sequential(n, start=0):
    return np.arange(start, start + n, dtype=np.int64)

def gen_uniform(n):
    return rng.integers(0, 2**32 - 1, size=n, dtype=np.int64)

def gen_zipfian(n):
    return np.minimum(rng.zipf(1.2, size=n), 2**31 - 1).astype(np.int64)

def gen_clustered(n):
    clusters = [(0, 1000), (100000, 101000), (5000000, 5001000)]
    choices = rng.integers(0, len(clusters), size=n)
    out = np.empty(n, dtype=np.int64)
    for i, (lo, hi) in enumerate(clusters):
        mask = choices == i
        out[mask] = rng.integers(lo, hi, size=mask.sum())
    return out

segment_size = 20_000

segments = [
    ("low_entropy_repeating", gen_low_entropy(segment_size)),
    ("sequential_ids", gen_sequential(segment_size)),
    ("uniform_32bit", gen_uniform(segment_size)),
    ("zipfian_smallints", gen_zipfian(segment_size)),
    ("clustered_ranges", gen_clustered(segment_size)),
    ("sequential_ids", gen_sequential(segment_size, start=1_000_000)),
    ("uniform_32bit", gen_uniform(segment_size)),
    ("low_entropy_repeating", gen_low_entropy(segment_size)),
]

stream = np.concatenate([arr for _, arr in segments])
truth = np.concatenate([[label] * len(arr) for label, arr in segments])

len(stream), stream[:10], truth[:10]

## Window the stream and compute local metrics

In [ ]:
def entropy_from_counts(counts):
    counts = np.asarray(counts)
    counts = counts[counts > 0]
    if counts.size == 0:
        return 0.0
    probs = counts / counts.sum()
    return float(-(probs * np.log2(probs)).sum())

def approx_entropy(values, max_bins=256):
    values = np.asarray(values)
    unique = len(np.unique(values))
    bins = min(max_bins, max(2, unique))
    counts, _ = np.histogram(values, bins=bins)
    return entropy_from_counts(counts)

def digit_lengths(arr):
    return np.array([len(str(abs(int(x)))) for x in arr], dtype=np.int16)

def cache_window_reuse(arr, window=64):
    if len(arr) < window:
        return 0.0
    vals = []
    for start in range(0, len(arr) - window + 1, window):
        w = arr[start:start+window]
        vals.append(1.0 - len(np.unique(w)) / len(w))
    return float(np.mean(vals)) if vals else 0.0

def mode_label(labels):
    vals, counts = np.unique(labels, return_counts=True)
    return vals[np.argmax(counts)]

def window_metrics(arr, labels, window_id, start, end):
    w = arr[start:end]
    lab = labels[start:end]
    n = len(w)
    d = np.diff(w) if n > 1 else np.array([0])
    absd = np.abs(d.astype(float))
    dl = digit_lengths(w)

    unique_count = len(np.unique(w))
    repetition_ratio = 1.0 - unique_count / max(n, 1)
    locality_ratio = float(np.mean(absd <= 16)) if len(absd) else 0.0
    digit_transition_rate = float(np.mean(np.diff(dl) != 0)) if n > 1 else 0.0
    digit_entropy = entropy_from_counts(np.bincount(dl))
    reuse = cache_window_reuse(w, 64)

    branch_pressure = (
        0.45 * digit_transition_rate +
        0.35 * (1.0 - locality_ratio) +
        0.20 * (1.0 - reuse)
    )

    return {
        "window_id": window_id,
        "start": start,
        "end": end,
        "truth_regime": mode_label(lab),
        "n": n,
        "approx_entropy_bits": approx_entropy(w),
        "repetition_ratio": repetition_ratio,
        "locality_small_delta_ratio": locality_ratio,
        "digit_length_entropy": digit_entropy,
        "digit_length_transition_rate": digit_transition_rate,
        "cache_window_reuse_proxy": reuse,
        "branch_pressure_score": branch_pressure,
        "delta_abs_mean": float(np.mean(absd)) if len(absd) else 0.0,
    }

window_size = 5_000
rows = []
for i, start in enumerate(range(0, len(stream), window_size)):
    end = min(start + window_size, len(stream))
    rows.append(window_metrics(stream, truth, i, start, end))

windows = pd.DataFrame(rows)
windows.head()

## Adaptive policy model

This lightweight policy mirrors Notebook 07 but operates per window.

In [ ]:
def norm01(s):
    s = pd.Series(s).astype(float)
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - lo) / (hi - lo)

work = windows.copy()

work["entropy_norm"] = norm01(work["approx_entropy_bits"])
work["delta_norm"] = norm01(np.log10(work["delta_abs_mean"] + 1.0))
work["branch_norm"] = norm01(work["branch_pressure_score"])

work["simd_suitability"] = (
    0.35 * work["delta_norm"] +
    0.25 * work["entropy_norm"] +
    0.25 * (1.0 - work["branch_norm"]) +
    0.15 * (1.0 - work["cache_window_reuse_proxy"])
).clip(0, 1)

work["scalar_suitability"] = (
    0.35 * work["locality_small_delta_ratio"] +
    0.30 * work["cache_window_reuse_proxy"] +
    0.25 * (1.0 - work["branch_norm"]) +
    0.10 * (1.0 - work["entropy_norm"])
).clip(0, 1)

work["fragmentation_score"] = (
    0.45 * work["branch_norm"] +
    0.30 * (1.0 - work["locality_small_delta_ratio"]) +
    0.25 * (1.0 - work["cache_window_reuse_proxy"])
).clip(0, 1)

work["coherence_score"] = (
    0.45 * work["scalar_suitability"] +
    0.25 * (1.0 - work["fragmentation_score"]) +
    0.15 * work["repetition_ratio"] +
    0.15 * work["locality_small_delta_ratio"]
).clip(0, 1)

work["hardware_pressure_proxy"] = (
    0.55 * work["fragmentation_score"] +
    0.25 * work["branch_norm"] +
    0.20 * (1.0 - work["locality_small_delta_ratio"])
).clip(0, 1)

def select_policy(row):
    if row["hardware_pressure_proxy"] > 0.78:
        return "guarded_fallback"
    if row["coherence_score"] > 0.68 and row["scalar_suitability"] > row["simd_suitability"]:
        return "coherent_local"
    if row["simd_suitability"] > 0.55 and row["simd_suitability"] > row["scalar_suitability"]:
        return "simd"
    if row["scalar_suitability"] > 0.55:
        return "scalar"
    return "hybrid"

work["adaptive_policy"] = work.apply(select_policy, axis=1)

work[["window_id", "truth_regime", "adaptive_policy", "coherence_score", "fragmentation_score", "hardware_pressure_proxy"]].head(12)

## Simulate throughput and switching costs

The values below are toy throughput units intended to make policy behavior visible.
Real benchmark integration can replace these values later.

In [ ]:
# Base throughput table by true regime and policy.
throughput_table = {
    "low_entropy_repeating": {"scalar": 1650, "simd": 1400, "coherent_local": 1750, "guarded_fallback": 1200, "hybrid": 1500},
    "sequential_ids": {"scalar": 1350, "simd": 1450, "coherent_local": 1300, "guarded_fallback": 1100, "hybrid": 1400},
    "uniform_32bit": {"scalar": 1100, "simd": 1900, "coherent_local": 1000, "guarded_fallback": 1200, "hybrid": 1650},
    "zipfian_smallints": {"scalar": 1200, "simd": 1500, "coherent_local": 1250, "guarded_fallback": 1150, "hybrid": 1550},
    "clustered_ranges": {"scalar": 900, "simd": 950, "coherent_local": 850, "guarded_fallback": 1200, "hybrid": 1050},
}

def policy_throughput(true_regime, policy):
    return throughput_table.get(true_regime, {}).get(policy, 1000)

work["adaptive_throughput_raw"] = [
    policy_throughput(r, p) for r, p in zip(work["truth_regime"], work["adaptive_policy"])
]

# Fixed baselines
work["fixed_scalar_throughput"] = [policy_throughput(r, "scalar") for r in work["truth_regime"]]
work["fixed_simd_throughput"] = [policy_throughput(r, "simd") for r in work["truth_regime"]]

# Switching cost: subtract when policy changes from previous window.
switching_cost_fraction = 0.03
policy_changed = work["adaptive_policy"].ne(work["adaptive_policy"].shift(1)).fillna(False)
work["policy_changed"] = policy_changed
work["adaptive_throughput_with_switch_cost"] = work["adaptive_throughput_raw"] * (
    1.0 - switching_cost_fraction * policy_changed.astype(float)
)

work["best_fixed_baseline"] = np.maximum(work["fixed_scalar_throughput"], work["fixed_simd_throughput"])
work["adaptive_gain_vs_best_fixed"] = (
    work["adaptive_throughput_with_switch_cost"] - work["best_fixed_baseline"]
)
work["adaptive_gain_pct_vs_best_fixed"] = 100.0 * work["adaptive_gain_vs_best_fixed"] / work["best_fixed_baseline"]

work[[
    "window_id", "truth_regime", "adaptive_policy", "policy_changed",
    "adaptive_throughput_with_switch_cost", "fixed_scalar_throughput",
    "fixed_simd_throughput", "adaptive_gain_pct_vs_best_fixed"
]].head(12)

## Export streaming adaptation table

In [ ]:
csv_path = RESULTS_DIR / "notebook08_streaming_runtime_adaptation.csv"
json_path = RESULTS_DIR / "notebook08_streaming_runtime_adaptation.json"

work.to_csv(csv_path, index=False)
work.to_json(json_path, orient="records", indent=2)

print("Saved:", csv_path)
print("Saved:", json_path)

## Figure 1 — Regime timeline

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook08_regime_timeline.png"

# Encode regimes for plotting
labels = sorted(work["truth_regime"].unique())
label_to_id = {lab: i for i, lab in enumerate(labels)}
y = work["truth_regime"].map(label_to_id)

plt.figure(figsize=(12, 4))
plt.step(work["window_id"], y, where="mid")
plt.yticks(list(label_to_id.values()), list(label_to_id.keys()))
plt.xlabel("Window")
plt.ylabel("True regime")
plt.title("Streaming Input: Regime Timeline")
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — Adaptive policy timeline

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook08_policy_timeline.png"

policies = sorted(work["adaptive_policy"].unique())
policy_to_id = {p: i for i, p in enumerate(policies)}
y = work["adaptive_policy"].map(policy_to_id)

plt.figure(figsize=(12, 4))
plt.step(work["window_id"], y, where="mid")
plt.yticks(list(policy_to_id.values()), list(policy_to_id.keys()))
plt.xlabel("Window")
plt.ylabel("Adaptive policy")
plt.title("Adaptive Execution Policy Timeline")
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Throughput comparison over time

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook08_throughput_timeline.png"

plt.figure(figsize=(12, 5))
plt.plot(work["window_id"], work["fixed_scalar_throughput"], label="fixed scalar")
plt.plot(work["window_id"], work["fixed_simd_throughput"], label="fixed SIMD")
plt.plot(work["window_id"], work["adaptive_throughput_with_switch_cost"], label="adaptive")
plt.xlabel("Window")
plt.ylabel("Throughput (toy units)")
plt.title("Streaming Runtime Adaptation: Throughput Timeline")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — Coherence and hardware pressure over time

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook08_coherence_pressure_timeline.png"

plt.figure(figsize=(12, 5))
plt.plot(work["window_id"], work["coherence_score"], label="coherence")
plt.plot(work["window_id"], work["hardware_pressure_proxy"], label="hardware pressure")
plt.xlabel("Window")
plt.ylabel("Score")
plt.title("Runtime Structure: Coherence vs Hardware Pressure")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Figure 5 — Adaptive gain by regime

In [ ]:
fig_path_5 = FIGURES_DIR / "notebook08_adaptive_gain_by_regime.png"

gain = (
    work.groupby("truth_regime", as_index=False)
    .agg(mean_gain_pct=("adaptive_gain_pct_vs_best_fixed", "mean"))
    .sort_values("mean_gain_pct")
)

plt.figure(figsize=(9, 5))
plt.bar(gain["truth_regime"], gain["mean_gain_pct"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Mean adaptive gain vs best fixed (%)")
plt.title("Adaptive Runtime Selection: Gain by Regime")
plt.tight_layout()
plt.savefig(fig_path_5, dpi=160)
plt.show()

print("Saved:", fig_path_5)

## Lab-report summary

In [ ]:
report_path = REPORTS_DIR / "report_08_streaming_runtime_adaptation.md"

summary = {
    "windows": int(len(work)),
    "policy_switches": int(work["policy_changed"].sum()),
    "mean_fixed_scalar": float(work["fixed_scalar_throughput"].mean()),
    "mean_fixed_simd": float(work["fixed_simd_throughput"].mean()),
    "mean_adaptive": float(work["adaptive_throughput_with_switch_cost"].mean()),
    "mean_gain_pct_vs_best_fixed": float(work["adaptive_gain_pct_vs_best_fixed"].mean()),
}

policy_counts = work["adaptive_policy"].value_counts().rename_axis("policy").reset_index(name="count")
regime_policy = pd.crosstab(work["truth_regime"], work["adaptive_policy"])

lines = [
    "# Report 08 — Streaming Runtime Adaptation",
    "",
    "This report simulates adaptive execution selection over a regime-switching integer stream.",
    "",
    "Constraint view:",
    "> runtime adaptation asks whether coherence persists as distributions drift.",
    "",
    "## Generated outputs",
    "",
    f"- Metrics CSV: `{csv_path}`",
    f"- Metrics JSON: `{json_path}`",
    f"- Figure: `{fig_path_1}`",
    f"- Figure: `{fig_path_2}`",
    f"- Figure: `{fig_path_3}`",
    f"- Figure: `{fig_path_4}`",
    f"- Figure: `{fig_path_5}`",
    "",
    "## Summary",
    "",
    pd.DataFrame([summary]).to_markdown(index=False),
    "",
    "## Policy counts",
    "",
    policy_counts.to_markdown(index=False),
    "",
    "## Regime × policy table",
    "",
    regime_policy.to_markdown(),
    "",
    "## Interpretation",
    "",
    "- The adaptive selector tracks changes in local distribution structure across windows.",
    "- Switching costs prevent unrealistic free adaptation.",
    "- Coherence and hardware-pressure timelines expose when the stream becomes stable or fragmented.",
    "- Adaptive gain is most meaningful in mixed or high-pressure regimes where fixed scalar/SIMD policies are brittle.",
    "",
    "## Next step",
    "",
    "Notebook 09 can test cross-hardware policy portability: does the same selector transfer across x86, ARM, AVX2, AVX512, and cloud baselines?",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if you are running this notebook in Google Colab and want to download generated outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook08_streaming_runtime_adaptation_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook08_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_08_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))